In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic questions

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40247608,0,What causes a significant portion of individua...
1,40247608,1,Is Fragile X syndrome considered a type of aut...
2,40247608,2,What causes some individuals with autism to ex...
3,40247608,3,What causes the unique communication styles an...
4,40247608,4,What are the main differences between Fragile ...
...,...,...,...
495,40933686,0,What challenges do autistic individuals face w...
496,40933686,1,Can we do away with the stigma surrounding aut...
497,40933686,2,What strategies do universities need to implem...
498,40933686,3,What challenges do autistic individuals face w...


# Demo RAG

In [4]:
# demonstrate RAG for one question
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)
print()
print(datetime.now())
demo_answer = rag.rag(demo_query, \
do_vector_search=False, num_results=2, model_handle_llm='llama3.2:1b', seed=42)
print(demo_answer)
print(datetime.now())

What causes a significant portion of individuals with autism spectrum disorders to have Fragile X syndrome as their underlying genetic condition?

2025-09-13 02:03:30.828108
Based on the context provided by the papers from PubMed, it appears that Fragile X syndrome is a significant underlying genetic condition for many individuals with autism spectrum disorders (ASD). 

One of the studies highlights that "Fragile X syndrome is the most common cause of intellectual disability and is often associated with developmental delays and behavioral challenges in ASD" (pmid: 40821657). This suggests that Fragile X syndrome can contribute to the development of ASD, particularly when combined with other genetic factors.

Another study explores the impact of age and sex on sensory sensitivities in autistic adults, with a focus on middle-aged and older individuals. While it does not specifically mention Fragile X syndrome as an underlying condition for ASD, the authors highlight that "autistic people

# Generate answers by RAG

## Function

In [5]:
# common parameters for RAG
print(rag.config['do_vector_search'])
print(rag.config['num_results'])

False
5


In [6]:
def generate_answers(synth_records, model_handle_llm, seed):
    answers = []
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        answer = rag.rag(query, \
        do_vector_search=rag.config['do_vector_search'], num_results=rag.config['num_results'], \
        model_handle_llm=model_handle_llm, seed=seed)
        pmid = record['pmid']
        ollama_seed_question = record['ollama_seed']
        answer_dict = {'pmid' : pmid, 'ollama_seed' : ollama_seed_question, \
        'answer_'+model_handle_llm : answer}
        answers.append(answer_dict)
    return pd.DataFrame.from_records(answers)

## Work

In [7]:
# sample size, dictionary format for dataset
sample_size = 30
synth_records = df_synth.to_dict('records')[:sample_size]
print(len(synth_records))

30


In [8]:
# generate answers for llama3.2
print(datetime.now())
answers_llama = generate_answers(synth_records=synth_records, \
model_handle_llm='llama3.2:1b', seed=42)
print(datetime.now())
answers_llama

2025-09-13 02:04:01.920863


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-13 02:31:35.764245


,pmid,ollama_seed,answer_llama3.2:1b
0,40247608,0,"Based on the provided context, a significant p..."
1,40247608,1,Based on the provided context and papers from ...
2,40247608,2,Based on the provided context and papers from ...
3,40247608,3,Based on the context provided by the papers fr...
4,40247608,4,"Based on the provided context, which includes ..."
5,40267907,0,This text appears to be a collection of scient...
6,40267907,1,"Based on the provided context, which includes ..."
7,40267907,2,Based on the provided context and papers from ...
8,40267907,3,The primary difference between individuals wit...
9,40267907,4,"Based on the provided PubMed articles, I found..."


In [9]:
# how long are the answers for llama3.2?
print(answers_llama['answer_llama3.2:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    310.00
0.75    362.75
0.99    442.62
1.00    449.00
Name: answer_llama3.2:1b, dtype: float64


In [10]:
# generate answers for gemma3
print(datetime.now())
answers_gemma = generate_answers(synth_records=synth_records, \
model_handle_llm='gemma3:1b', seed=42)
print(datetime.now())
answers_gemma

2025-09-13 02:31:35.813482


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-13 02:59:40.225795


,pmid,ollama_seed,answer_gemma3:1b
0,40247608,0,"Okay, based on the provided context and the re..."
1,40247608,1,"Okay, based on the provided text, here’s the a..."
2,40247608,2,"Okay, let's analyze the provided text and answ..."
3,40247608,3,"Okay, based on the provided text and the PubMe..."
4,40247608,4,"Okay, let's analyze the provided context and a..."
5,40267907,0,"Okay, here's an analysis of the provided text,..."
6,40267907,1,"Okay, here's an answer based on the provided c..."
7,40267907,2,"Okay, based on the provided text and the quest..."
8,40267907,3,"Okay, let’s analyze the provided text and answ..."
9,40267907,4,"Okay, let’s analyze these papers based on the ..."


In [11]:
# how long are the answers for gemma3?
print(answers_gemma['answer_gemma3:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    399.50
0.75    480.25
0.99    701.56
1.00    741.00
Name: answer_gemma3:1b, dtype: float64


In [12]:
# put together answers from different models
df_synth_answer = answers_llama.merge(answers_gemma, on=['pmid', 'ollama_seed'], how='inner')
df_synth_answer

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40247608,0,"Based on the provided context, a significant p...","Okay, based on the provided context and the re..."
1,40247608,1,Based on the provided context and papers from ...,"Okay, based on the provided text, here’s the a..."
2,40247608,2,Based on the provided context and papers from ...,"Okay, let's analyze the provided text and answ..."
3,40247608,3,Based on the context provided by the papers fr...,"Okay, based on the provided text and the PubMe..."
4,40247608,4,"Based on the provided context, which includes ...","Okay, let's analyze the provided context and a..."
5,40267907,0,This text appears to be a collection of scient...,"Okay, here's an analysis of the provided text,..."
6,40267907,1,"Based on the provided context, which includes ...","Okay, here's an answer based on the provided c..."
7,40267907,2,Based on the provided context and papers from ...,"Okay, based on the provided text and the quest..."
8,40267907,3,The primary difference between individuals wit...,"Okay, let’s analyze the provided text and answ..."
9,40267907,4,"Based on the provided PubMed articles, I found...","Okay, let’s analyze these papers based on the ..."


# Write CSV file

In [13]:
# write CSV file
df_synth_answer.to_csv('../data/data-synth-answer.csv', index=False, sep='\t')

In [14]:
print(datetime.now())

2025-09-13 02:59:40.295095
